# supra50m → Mercury-2 diffusion LM **by the translator C alone** (run 4: quality paths)

**Contract.** Model B is *never trained*. `B* = C(supra50m)` = the donor's own weights +
per-block SVD-frame corrections **emitted by C** + an emitted [MASK] embedding. C is a
per-block weight-tied hypernet trained through the diffusion loss on the **self-zoo**
(supra's own depth-truncated sub-stacks); all data is **sampled from supra itself**.

**Run-3 was positive** (B* beats the floor at every mask rate; the shuffled-signature
control separates 8.44 vs 4.97 ⇒ C genuinely reads per-block weights). Run-4 attacks the
remaining quality gap along the five paths from `DECISION_GRAPH.md`:
**(1) corpus v2** — unigram-prompted self-generation (diverse, still donor-only);
**(2) sampler** — semi-AR block diffusion + cosine/Gumbel/remask/temp-anneal, more steps;
**(3) zoo+L=11** (full L=12 still held out); **(4) capacity 64×64**; **(5) KD soft targets**
from the AR teacher at masked positions (= the allowed 'Q-A captured from the model').

**Setup.** GPU + Internet, Run All (~35 min). Saves `B*` to `/kaggle/working`.

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import glob, json, math, time, torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors.torch import load_file, save_file

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
MODEL_ID = 'SupraLabs/Supra-50M-Instruct'

def resolve_model():
    env = os.environ.get('CKPT_DIR')
    if env and os.path.exists(os.path.join(env, 'config.json')): return env
    ds = glob.glob('/kaggle/input/**/config.json', recursive=True)
    if ds: return os.path.dirname(ds[0])
    from huggingface_hub import snapshot_download
    return snapshot_download(MODEL_ID, allow_patterns=['config.json', 'model.safetensors', 'tokenizer.json'])

MODEL_DIR = resolve_model()
N_GEN     = 1024
N_HELD    = 64
SEQ_LEN   = 256
PROMPT_LEN = 4                    # corpus v2: unigram-sampled prompt tokens (dropped after)
N_BOOT    = 128                   # bootstrap seqs for the unigram stats
TRUNC_DEPTHS = [2, 4, 6, 8, 10, 11]   # self-zoo; the FULL L=12 stack stays held out
C_STEPS   = 6000
C_BS      = 8
SUB       = 64                    # covariant correction subspace (A is SUB x SUB)
SIG_K     = 32
D_Z       = 16
EPS_T     = 0.05
KD_LAMBDA = 0.3                   # weight of AR-teacher soft targets at masked positions
print('device', DEV, '| model', MODEL_DIR)

In [ ]:
tj = json.load(open(os.path.join(MODEL_DIR, 'tokenizer.json')))
inv_vocab = {i: t for t, i in tj['model']['vocab'].items()}
def decode(ids):
    return ''.join(inv_vocab.get(int(i), '?') for i in ids).replace('\u2581', ' ') \
             .replace('\u0120', ' ').replace('\u010a', '\n')

CFG = json.load(open(os.path.join(MODEL_DIR, 'config.json')))
H, KV = CFG['num_attention_heads'], CFG.get('num_key_value_heads', CFG['num_attention_heads'])
D = CFG['hidden_size']; HD = CFG.get('head_dim', D // H); FF = CFG['intermediate_size']
EPS = CFG.get('rms_norm_eps', 1e-5)
rp = CFG.get('rope_parameters') or {}
THETA = CFG.get('rope_theta', rp.get('rope_theta', 10000))
V = CFG['vocab_size']; MASK_ID = V
PROJ = ('self_attn.q_proj', 'self_attn.k_proj', 'self_attn.v_proj', 'self_attn.o_proj',
        'mlp.gate_proj', 'mlp.up_proj', 'mlp.down_proj')

def rotate_half(x):
    d = x.shape[-1] // 2
    return torch.cat([-x[..., d:], x[..., :d]], dim=-1)
def rope(x, pos):
    inv = 1.0 / (THETA ** (torch.arange(0, HD, 2, device=DEV).float() / HD))
    fr = pos[:, None].float() * inv[None, :]
    cos = torch.cat([fr.cos(), fr.cos()], -1)[None, None]
    sin = torch.cat([fr.sin(), fr.sin()], -1)[None, None]
    return x * cos + rotate_half(x) * sin
def rms(x, w): return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + EPS) * w

def llama_forward(w, idx, L, causal=True, mask_row=None):
    B, T = idx.shape
    pos = torch.arange(T, device=DEV)
    E = w['model.embed_tokens.weight']
    x = E[idx.clamp_max(V - 1)]
    if mask_row is not None:
        x = torch.where((idx == MASK_ID)[..., None], mask_row, x)
    bias = torch.full((T, T), float('-inf'), device=DEV).triu(1)[None, None] if causal else None
    for l in range(L):
        p = lambda n: w[f'model.layers.{l}.{n}.weight']
        h = rms(x, p('input_layernorm'))
        q = (h @ p('self_attn.q_proj').T).view(B, T, H, HD).transpose(1, 2)
        k = (h @ p('self_attn.k_proj').T).view(B, T, KV, HD).transpose(1, 2)
        v = (h @ p('self_attn.v_proj').T).view(B, T, KV, HD).transpose(1, 2)
        q, k = rope(q, pos), rope(k, pos)
        k, v = k.repeat_interleave(H // KV, 1), v.repeat_interleave(H // KV, 1)
        att = (q @ k.transpose(-1, -2)) / (HD ** 0.5)
        if bias is not None: att = att + bias
        o = (att.softmax(-1) @ v).transpose(1, 2).reshape(B, T, D)
        x = x + o @ p('self_attn.o_proj').T
        h2 = rms(x, p('post_attention_layernorm'))
        x = x + (F.silu(h2 @ p('mlp.gate_proj').T) * (h2 @ p('mlp.up_proj').T)) @ p('mlp.down_proj').T
    return rms(x, w['model.norm.weight']) @ E.T

@torch.no_grad()
def generate_ar(w, L, prompt, n, temperature=0.9, top_k=40):
    """KV-cached AR sampling (only to capture the donor's own data)."""
    B = prompt.shape[0]; E = w['model.embed_tokens.weight']
    kc, vc = [None] * L, [None] * L
    def step(tokens, off):
        T = tokens.shape[1]; pos = torch.arange(off, off + T, device=DEV)
        x = E[tokens]
        for l in range(L):
            p = lambda n: w[f'model.layers.{l}.{n}.weight']
            h = rms(x, p('input_layernorm'))
            q = (h @ p('self_attn.q_proj').T).view(B, T, H, HD).transpose(1, 2)
            k = (h @ p('self_attn.k_proj').T).view(B, T, KV, HD).transpose(1, 2)
            v = (h @ p('self_attn.v_proj').T).view(B, T, KV, HD).transpose(1, 2)
            q, k = rope(q, pos), rope(k, pos)
            if kc[l] is not None:
                k = torch.cat([kc[l], k], 2); v = torch.cat([vc[l], v], 2)
            kc[l], vc[l] = k, v
            kk, vv = k.repeat_interleave(H // KV, 1), v.repeat_interleave(H // KV, 1)
            att = (q @ kk.transpose(-1, -2)) / (HD ** 0.5)
            Tt = k.shape[2]; kpos = torch.arange(Tt, device=DEV)[None, :]
            att = att + torch.where(kpos <= pos[:, None], 0.0, float('-inf'))
            o = (att.softmax(-1) @ vv).transpose(1, 2).reshape(B, T, D)
            x = x + o @ p('self_attn.o_proj').T
            h2 = rms(x, p('post_attention_layernorm'))
            x = x + (F.silu(h2 @ p('mlp.gate_proj').T) * (h2 @ p('mlp.up_proj').T)) @ p('mlp.down_proj').T
        return (rms(x, w['model.norm.weight']) @ E.T)[:, -1, :]
    logits = step(prompt, 0); out = prompt
    for _ in range(n):
        lo = logits / max(temperature, 1e-6)
        vt, _ = torch.topk(lo, top_k); lo[lo < vt[:, [-1]]] = float('-inf')
        nxt = torch.multinomial(lo.softmax(-1), 1)
        out = torch.cat([out, nxt], 1)
        logits = step(nxt, out.shape[1] - 1)
    return out

supra = {k: v.float().to(DEV) for k, v in load_file(os.path.join(MODEL_DIR, 'model.safetensors')).items()}
L_SUPRA = CFG['num_hidden_layers']
print(f'supra50m: L={L_SUPRA} d={D} V={V}')

In [ ]:
# Path 1 (decision graph): bare-BOS self-generation is degenerate ('as as as ...') -- a
# 50M model talking to itself from nothing. Fix WITHOUT external data: bootstrap unigram
# stats from a bare batch, then seed generation with short prompts sampled from the donor's
# own unigram distribution (random noise tokens, not data) and keep only the continuations.
t0 = time.time()
boot = generate_ar(supra, L_SUPRA, torch.full((N_BOOT, 1), 1, dtype=torch.long, device=DEV),
                   SEQ_LEN // 2, temperature=1.0)[:, 1:]
uni = torch.bincount(boot.reshape(-1), minlength=V).float() + 1e-3
uni = uni / uni.sum()

chunks, need, gb = [], N_GEN + N_HELD, 64
temps = [0.7, 0.9, 1.0]
while sum(c.shape[0] for c in chunks) < need:
    tt = temps[len(chunks) % len(temps)]
    prompt = torch.multinomial(uni.expand(gb, V), PROMPT_LEN, replacement=True)
    full = generate_ar(supra, L_SUPRA, prompt, SEQ_LEN, temperature=tt)
    chunks.append(full[:, PROMPT_LEN:])              # keep continuations only
corpus = torch.cat(chunks, 0)[:need]
train_ids, held_ids = corpus[:N_GEN], corpus[N_GEN:]
print(f'corpus v2: train {tuple(train_ids.shape)} held {tuple(held_ids.shape)} in {time.time()-t0:.0f}s')
print('sample:', repr(decode(train_ids[0, :48])))

In [ ]:
# Self-zoo: depth-truncated sub-stacks of supra itself (layers 0..L-1 + final norm + tied
# head). Run-4 adds L=11 (the run-3 probe localized damage to unseen blocks 10-11); the
# full L=12 composition remains the held-out extrapolation claim.
for Lz in TRUNC_DEPTHS:
    ce = F.cross_entropy(llama_forward(supra, held_ids[:8, :-1], Lz).reshape(-1, V),
                         held_ids[:8, 1:].reshape(-1)).item()
    print(f'sub-stack L={Lz:>2}: held AR CE {ce:5.2f}')
print('self-zoo ready (no training needed)')

In [ ]:
def sample_mask_rate(b): return EPS_T + (1.0 - EPS_T) * torch.rand(b, device=DEV)
def forward_mask(x0, t):
    B, L = x0.shape; noise = torch.rand(B, L, device=x0.device); m = noise < t[:, None].expand(B, L)
    empty = ~m.any(dim=1)
    if empty.any(): m[empty.nonzero(as_tuple=True)[0], noise[empty].argmin(dim=1)] = True
    return torch.where(m, torch.full_like(x0, MASK_ID), x0), m
def diffusion_loss(logits, x0, m, t):
    B, L, Vv = logits.shape
    ce = F.cross_entropy(logits.reshape(-1, Vv), x0.reshape(-1), reduction='none').view(B, L)
    return ((ce * m).sum(dim=1) / (t * L)).mean()

@torch.no_grad()
def denoise(fn, ids, frozen, steps=64, temperature=0.0):
    """run-3 sampler (linear reveal, fixed temperature) -- kept as the comparison point."""
    B, L = ids.shape
    for s in range(steps):
        masked = (ids == MASK_ID) & ~frozen; n_left = int(masked.sum().item())
        if n_left == 0: break
        logits = fn(ids)
        probs = (logits / temperature).softmax(-1) if temperature > 0 else logits.softmax(-1)
        pred = torch.multinomial(probs.view(-1, V), 1).view(B, L) if temperature > 0 else probs.argmax(-1)
        conf = probs.max(-1).values.masked_fill(~masked, float('-inf'))
        k = min(max(1, n_left // (steps - s)), n_left)
        idxs = conf.view(-1).topk(k).indices
        ids.view(-1)[idxs] = pred.view(-1)[idxs]
    masked = (ids == MASK_ID) & ~frozen
    if masked.any(): ids = torch.where(masked, fn(ids).argmax(-1), ids)
    return ids

@torch.no_grad()
def denoise_v2(fn, ids, frozen, steps=128, temp0=0.9, alg_temp=0.5, remask_frac=0.15):
    """run-4 sampler: cosine reveal schedule (MaskGIT), Gumbel noise on the confidence
    ranking (Dream), temperature annealed to argmax, and remasking of low-confidence
    already-revealed tokens during the first half (lets early mistakes be revised)."""
    B, L = ids.shape
    n0 = int(((ids == MASK_ID) & ~frozen).sum().item())
    if n0 == 0: return ids
    for s in range(steps):
        masked = (ids == MASK_ID) & ~frozen
        n_left = int(masked.sum().item())
        if n_left == 0: break
        logits = fn(ids)
        tau = temp0 * max(0.0, 1.0 - s / max(1, steps - 1))   # anneal -> argmax
        if tau > 0.05:
            probs = (logits / tau).softmax(-1)
            pred = torch.multinomial(probs.view(-1, V), 1).view(B, L)
        else:
            probs = logits.softmax(-1); pred = probs.argmax(-1)
        conf = probs.max(-1).values.masked_fill(~masked, float('-inf'))
        if alg_temp > 0:                                        # stochastic reveal order
            g = torch.rand_like(conf).clamp_min(1e-9)
            conf = conf + alg_temp * (-(-g.log()).log()) * (s < steps // 2)
        keep = math.cos(math.pi / 2 * (s + 1) / steps)          # cosine: reveal slow->fast
        k = max(1, min(n_left, n_left - int(n0 * keep)))
        idxs = conf.view(-1).topk(k).indices
        ids.view(-1)[idxs] = pred.view(-1)[idxs]
        if remask_frac > 0 and s < steps // 2:                  # revise weak early choices
            revealed = (ids != MASK_ID) & ~frozen
            if revealed.any():
                cur = probs.gather(-1, ids.clamp_max(V - 1)[..., None]).squeeze(-1)
                cur = cur.masked_fill(~revealed, float('inf'))
                q = max(1, int(revealed.sum().item() * remask_frac))
                weak = (-cur.view(-1)).topk(q).indices
                ids.view(-1)[weak] = MASK_ID
    masked = (ids == MASK_ID) & ~frozen
    if masked.any(): ids = torch.where(masked, fn(ids).argmax(-1), ids)
    return ids

@torch.no_grad()
def semi_ar_generate(fn, prompt, total_len, block=32, steps_per_block=32, **kw):
    """LLaDA-style semi-autoregressive block diffusion: generate left->right in blocks,
    each block denoised in parallel conditioned on everything before it. Plays directly to
    B*'s inherited AR (left-context) strength."""
    ids = prompt.clone()
    while ids.shape[1] < total_len:
        b = min(block, total_len - ids.shape[1])
        win = torch.cat([ids, torch.full((ids.shape[0], b), MASK_ID, dtype=torch.long, device=DEV)], 1)
        frozen = torch.zeros_like(win, dtype=torch.bool); frozen[:, :ids.shape[1]] = True
        ids = denoise_v2(fn, win, frozen, steps=steps_per_block, **kw)
    return ids

@torch.no_grad()
def masked_ce_at(fn, ids, t_val, n=32):
    x0 = ids[:n]; x_t, m = forward_mask(x0, torch.full((x0.shape[0],), t_val, device=DEV))
    lo = fn(x_t); ce = F.cross_entropy(lo.reshape(-1, V), x0.reshape(-1), reduction='none').view(x0.shape)
    return (ce * m).sum().item() / m.sum().item()
print('diffusion core + run-4 samplers ready')

In [ ]:
@torch.no_grad()
def svd_cache(w, L):
    """Per layer, per projection: (U_s [o x SUB], V_s [i x SUB], signature)."""
    cache = []
    for l in range(L):
        layer = {}
        for n in PROJ:
            W = w[f'model.layers.{l}.{n}.weight']
            U, S, Vh = torch.linalg.svd(W, full_matrices=False)
            sig = torch.cat([(S / (S.norm() + 1e-9))[:SIG_K],
                             torch.log(W.norm() + 1e-9)[None]])
            layer[n] = (U[:, :SUB].contiguous(), Vh[:SUB].T.contiguous(), sig)
        cache.append(layer)
    return cache

SIG_DIM = (SIG_K + 1) * len(PROJ)

class TranslatorC(nn.Module):
    """Per-block weight-tied hypernet. Emits, per projection, a SUB x SUB correction A(z)
    in the matrix's OWN top-SUB SVD frame (input invariant, output covariant), plus norm-
    gain scalars and a frame-free [MASK] embedding (softmax combo of the donor's own rows).
    Deltas init 0 => B starts exactly at the raw-bidirectional donor."""
    def __init__(self, d_z=D_Z, sub=SUB, h=128):
        super().__init__(); self.sub = sub
        self.enc = nn.Sequential(nn.Linear(SIG_DIM + 1, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, d_z))
        self.A = nn.ParameterDict({n.replace('.', '_'): nn.Parameter(torch.zeros(sub * sub, d_z)) for n in PROJ})
        self.Mg = nn.Parameter(torch.zeros(2, d_z))
        self.mask_logits = nn.Parameter(torch.zeros(V))
    def mask_row(self, w): return self.mask_logits.softmax(0) @ w['model.embed_tokens.weight']
    def emit(self, w, L, cache):
        out = dict(w)
        for l in range(L):
            sig = torch.cat([cache[l][n][2] for n in PROJ])
            z = self.enc(torch.cat([sig, sig.new_tensor([0.0 if L == 1 else l / (L - 1)])]))
            for n in PROJ:
                U, Vs, _ = cache[l][n]
                A = (self.A[n.replace('.', '_')] @ z).view(self.sub, self.sub)
                out[f'model.layers.{l}.{n}.weight'] = w[f'model.layers.{l}.{n}.weight'] + U @ A @ Vs.T
            dg = (self.Mg @ z)
            out[f'model.layers.{l}.input_layernorm.weight'] = w[f'model.layers.{l}.input_layernorm.weight'] * (1 + dg[0])
            out[f'model.layers.{l}.post_attention_layernorm.weight'] = w[f'model.layers.{l}.post_attention_layernorm.weight'] * (1 + dg[1])
        return out

t0 = time.time()
supra_cache = svd_cache(supra, L_SUPRA)
C = TranslatorC().to(DEV)
print(f'SVD cached + C built ({sum(p.numel() for p in C.parameters())} params, {time.time()-t0:.0f}s)')

In [ ]:
# Path 5: at masked positions, mix the one-hot target with the AR teacher's distribution
# P_supra(x_t | x_<t) -- dense donor knowledge, and literally the allowed 'Q-A captured
# from the model' (context -> teacher answer distribution). Teacher runs no-grad.
opt = torch.optim.AdamW(C.parameters(), lr=3e-4)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 100))
ema, t0 = None, time.time()
for step in range(1, C_STEPS + 1):
    Lz = TRUNC_DEPTHS[step % len(TRUNC_DEPTHS)]
    x0 = train_ids[torch.randint(0, train_ids.shape[0], (C_BS,), device=DEV)]
    t = sample_mask_rate(C_BS); x_t, m = forward_mask(x0, t)
    wB = C.emit(supra, Lz, supra_cache)
    lo = llama_forward(wB, x_t, Lz, causal=False, mask_row=C.mask_row(supra))
    loss = diffusion_loss(lo, x0, m, t)
    if KD_LAMBDA > 0:
        with torch.no_grad():
            soft = llama_forward(supra, x0, L_SUPRA, causal=True)[:, :-1].softmax(-1)
        lsm = lo[:, 1:].log_softmax(-1)
        kd = -(soft * lsm).sum(-1)                       # CE vs teacher dist, positions 1..T-1
        kd = ((kd * m[:, 1:]).sum(1) / (t * x0.shape[1])).mean()
        loss = (1 - KD_LAMBDA) * loss + KD_LAMBDA * kd
    opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(C.parameters(), 1.0); opt.step(); sched.step()
    ema = loss.item() if ema is None else 0.98 * ema + 0.02 * loss.item()
    if step % 500 == 0 or step == 1:
        with torch.no_grad():
            fn = lambda i: llama_forward(C.emit(supra, 4, supra_cache), i, 4, causal=False, mask_row=C.mask_row(supra))
            hce = masked_ce_at(fn, held_ids, 0.5, n=16)
        print(f'step {step:>5}  train(ema) {ema:6.3f}  sub-stack-L4 masked-CE@0.5 {hce:5.2f}  ({time.time()-t0:.0f}s)')
print('C trained — B was never trained, only C was')

In [ ]:
with torch.no_grad():
    wB = C.emit(supra, L_SUPRA, supra_cache); mrow = C.mask_row(supra)
fnB = lambda i: llama_forward(wB, i, L_SUPRA, causal=False, mask_row=mrow)
mean_row = supra['model.embed_tokens.weight'].mean(0)
fn0 = lambda i: llama_forward(supra, i, L_SUPRA, causal=False, mask_row=mean_row)

ar = F.cross_entropy(llama_forward(supra, held_ids[:16, :-1], L_SUPRA).reshape(-1, V), held_ids[:16, 1:].reshape(-1)).item()
print(f'uniform ln(V) = {math.log(V):.2f} | donor AR next-token CE (held, corpus v2) = {ar:.2f}')
print('\nheld masked-CE:    floor(raw bidir)   B*=C(supra)')
for tv in (0.3, 0.5, 0.7, 0.9):
    print(f'  t={tv}:   {masked_ce_at(fn0, held_ids, tv):7.2f}      {masked_ce_at(fnB, held_ids, tv):7.2f}')

x0 = held_ids[:8]; x_c, m = forward_mask(x0, torch.full((x0.shape[0],), 0.25, device=DEV))
for name, fn in (('floor', fn0), ('B*', fnB)):
    rec = denoise_v2(fn, x_c.clone(), ~m, steps=32, temp0=0.0, alg_temp=0.0, remask_frac=0.0)
    acc = ((rec == x0) & m).sum().item() / m.sum().item()
    print(f'\nreconstruction ({name}): token accuracy {acc:.1%}')
    if name == 'B*':
        print('  original :', repr(decode(x0[0, :48]))); print('  recovered:', repr(decode(rec[0, :48])))

# ---- sampler comparison on B*: run-3 sampler vs tuned v2 vs semi-AR blocks ----
print('\n=== generation: sampler comparison (B*, 128 tokens) ===')
ids = torch.full((2, 128), MASK_ID, dtype=torch.long, device=DEV); fr = torch.zeros_like(ids, dtype=torch.bool)
g1 = denoise(fnB, ids.clone(), fr, steps=64, temperature=0.7)
print('\n[A] run-3 sampler (64 steps, temp .7):', repr(decode(g1[0])[:240]))
g2 = denoise_v2(fnB, ids.clone(), fr, steps=128, temp0=0.9, alg_temp=0.5, remask_frac=0.15)
print('\n[B] tuned v2 (128 steps, cosine+Gumbel+remask+anneal):', repr(decode(g2[0])[:240]))
bos = torch.full((2, 1), 1, dtype=torch.long, device=DEV)
g3 = semi_ar_generate(fnB, bos, 129, block=32, steps_per_block=32, temp0=0.9, alg_temp=0.5, remask_frac=0.15)
print('\n[C] semi-AR blocks (32x32 steps):', repr(decode(g3[0, 1:])[:240]))
# prompted continuation with the best mode
prompt = held_ids[1, :24][None].repeat(2, 1)
g4 = semi_ar_generate(fnB, prompt, 24 + 96, block=32, steps_per_block=32, temp0=0.9, alg_temp=0.5, remask_frac=0.15)
print('\nprompt    :', repr(decode(prompt[0])))
print('semi-AR continuation:', repr(decode(g4[0, 24:])))

# ---- controls/probes ----
with torch.no_grad():
    wB11 = C.emit(supra, 11, supra_cache)               # deltas on blocks 0..10 (seen)
fn11 = lambda i: llama_forward(wB11, i, L_SUPRA, causal=False, mask_row=mrow)
print(f'\nprobe — deltas on blocks 0..10 only: masked-CE@0.5 {masked_ce_at(fn11, held_ids, 0.5):.2f} '
      f'(vs all 12: {masked_ce_at(fnB, held_ids, 0.5):.2f})')
with torch.no_grad():
    perm = torch.randperm(L_SUPRA).tolist()
    wB_mm = C.emit(supra, L_SUPRA, [supra_cache[p] for p in perm])
fn_mm = lambda i: llama_forward(wB_mm, i, L_SUPRA, causal=False, mask_row=mrow)
print(f'control — shuffled-signature emit masked-CE@0.5: {masked_ce_at(fn_mm, held_ids, 0.5):.2f} '
      f'(vs B*; equal ⇒ C ignores signatures)')

In [ ]:
out_sd = {k: v.detach().cpu().contiguous() for k, v in wB.items()}
out_sd['mercury.mask_embedding'] = mrow.detach().cpu().contiguous()
path = '/kaggle/working/supra_mercury2_by_C.safetensors' if os.path.isdir('/kaggle/working') else 'supra_mercury2_by_C.safetensors'
save_file(out_sd, path)
print('saved B* = C(supra50m) ->', path, f'({os.path.getsize(path)/1e6:.0f} MB)')

## How to read
- **Corpus v2 changes the scale of all CE numbers** (more diverse text ⇒ higher donor AR CE
  than run-3's 1.51); compare B* to the floor and the donor AR CE *within this run*.
- **Sampler comparison [A]/[B]/[C]**: [C] semi-AR blocks should be the most fluent — it
  plays to the AR strength B* inherits from the donor. [B] vs [A] isolates the schedule/
  remasking gains at equal architecture.
- **Probe 0..10-only vs all-12**: run-3 had 4.34 vs 4.97 (unseen blocks 10-11 hurt). With
  L=11 in the zoo, the gap should close; block 11 + full-depth composition remain held out.
- **Shuffled-signature control** must stay separated (run-3: 8.44 vs 4.97) — the proof that
  C reads per-block weights. B is never trained anywhere in this notebook.